# ObjectOracle — Exploratory Analysis & Evaluation
Visualize detections, evaluate model performance, inspect dataset statistics.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io, base64, json

API_BASE = 'http://localhost:8000'
print('ObjectOracle Notebook ready')

## 1. Health check

In [ ]:
health = requests.get(f'{API_BASE}/health').json()
print(json.dumps(health, indent=2))

## 2. Detect objects in a local image

In [ ]:
IMAGE_PATH = 'sample.jpg'  # change to your image

with open(IMAGE_PATH, 'rb') as f:
    resp = requests.post(
        f'{API_BASE}/detect',
        files={'file': f},
        params={'confidence': 0.35, 'scene': True}
    )

result = resp.json()
print(f"Detected {result['object_count']} objects in {result['inference_ms']}ms")
print(f"Scene: {result['scene_tags'][:2]}")

## 3. Visualize bounding boxes

In [ ]:
COLORS = plt.cm.tab10.colors
label_color_map = {}
color_idx = 0

def get_color(label):
    global color_idx
    if label not in label_color_map:
        label_color_map[label] = COLORS[color_idx % len(COLORS)]
        color_idx += 1
    return label_color_map[label]

img = Image.open(IMAGE_PATH)
fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.imshow(img)
ax.axis('off')

for obj in result['objects']:
    x1, y1, x2, y2 = obj['bbox']
    w, h = x2 - x1, y2 - y1
    color = get_color(obj['label'])
    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1 - 6, f"{obj['label']} {obj['confidence']:.0%}",
            color='white', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))

plt.title(f"ObjectOracle Detection — {result['object_count']} objects — {result['inference_ms']}ms", fontsize=12)
plt.tight_layout()
plt.savefig('detection_result.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Class distribution bar chart

In [ ]:
from collections import Counter

counts = Counter(obj['label'] for obj in result['objects'])
labels, values = zip(*sorted(counts.items(), key=lambda x: -x[1]))
colors = [get_color(l) for l in labels]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title('Detected class distribution', fontsize=13)
ax.set_ylabel('Count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Confidence score distribution

In [ ]:
confidences = [obj['confidence'] for obj in result['objects']]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(confidences, bins=20, color='#7c6ef5', edgecolor='white', linewidth=0.5)
ax.axvline(np.mean(confidences), color='#f97316', linestyle='--', label=f'Mean: {np.mean(confidences):.2f}')
ax.set_title('Confidence score distribution')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()